# Planning — Exercise 4

In Exercise 2 the agent decided each tool call on the fly — one step at a time.
For a simple move that is fine. For drawing a hexagon it is not:
the model has to hold the whole geometry in its head across 12 tool calls,
and any reasoning error compounds with every step.

**Plan-then-execute** splits the work into two phases:

```
Phase 1 — PLAN    : LLM produces a complete plan (JSON or code)
Phase 2 — EXECUTE : Python runs the plan — no LLM involved
```

This notebook covers two ways to produce that plan:

| Approach | How the plan is expressed | Schema needed? |
|---|---|---|
| **4b — JSON schema** | Structured data validated by Pydantic | Yes — define once per tool |
| **4c–4e — Code-as-actions** | Python script wrapped in XML tags | No |

Both approaches use **exactly one LLM call** to plan, then pure Python to execute.


## Setup — imports and tools

We import the same tool functions from `utils_03`.


In [1]:
import ast
import re
import math
from typing import Literal, Union

from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel
from llm_config import get_llm

from utils_03 import TOOLS, safe_invoke, MoveArgs, RotateArgs


# LangChain tool objects — for safe_invoke (has .invoke())
TOOL_OBJECTS = {t.name: t for t in TOOLS}
# Plain name → function dict used by execute_code() later
TOOL_FUNCTIONS = {t.name: t.func for t in TOOLS}

print("Tools available:", list(TOOL_FUNCTIONS.keys()))

/home/ubuntu/tutorial_rpa/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Tools registered:
  get_pose         — Return the turtle's current pose: x, y position and heading.
  move_forward     — Move the turtle forward along its current heading by an exact distance.
  rotate           — Rotate the turtle in place by a target angle. Does not move forward.
  set_pen          — Control the turtle's drawing pen: colour, width, and up/down state.
  teleport         — Teleport the turtle to an absolute position and heading instantly.
  reset            — Reset the turtle simulator, sending the turtle back to the origin and clearing any drawn lines.
Tools available: ['get_pose', 'move_forward', 'rotate', 'set_pen', 'teleport', 'reset']


## Exercise 4a — The cost of reactive planning

| Approach | LLM calls for a square | LLM calls for a hexagon |
|---|---|---|
| Reactive (Exercise 2) | 9+ — one per tool call | 15+ |
| Plan-then-execute (either approach) | **1** | **1** |

The reactive agent also accumulates a growing context window at every step.
Both plan-first approaches pay one fixed LLM cost up front, then execute in pure Python.


## Exercise 4b — Plan with a JSON schema

The first approach uses `llm.with_structured_output(Schema)` to force the model
to return data that matches a Pydantic model exactly.
The model never produces free text — it fills in the schema fields directly.

You define the schema once. After that, every call is guaranteed to return
a valid `DrawingPlan` object — no JSON parsing, no fence-stripping, no `.text`.

**Run the cell and inspect `plan`.** It is a Python object — `plan.steps` is a list
you can iterate, validate, and replay without touching a string.


In [2]:
# ── Define the schema ────────────────────────────────────────────────────────
class MoveStep(MoveArgs):
    tool: Literal["move_forward"]
    # distance_m: float

class RotateStep(RotateArgs):
    tool: Literal["rotate"]
    # angle_deg: float

class DrawingPlan(BaseModel):
    shape: str
    n_sides: int
    side_m: float
    steps: list[Union[MoveStep, RotateStep]]


# ── Ask the model to fill it in ───────────────────────────────────────────────
SCHEMA_SYSTEM = """You are a geometric planner for a TurtleSim robot.
The turtle operates in an 11 x 11 metre arena.
Given a drawing task, produce a step-by-step plan.
Alternate move_forward and rotate steps for each side of the shape.
Exterior angle = 360 / n_sides degrees, counter-clockwise (positive).
Use only the tools available, and ouput in the exact JSON format specified by the schema.
"""

llm = get_llm()
planner = llm.with_structured_output(DrawingPlan)

plan = planner.invoke([
    SystemMessage(SCHEMA_SYSTEM),
    HumanMessage("Draw a traingle with side length 2 metres."),
])

print(f"Shape  : {plan.shape}")
print(f"Sides  : {plan.n_sides}")
print(f"Side   : {plan.side_m} m")
print(f"Steps  : {len(plan.steps)}")
print()
for i, step in enumerate(plan.steps):
    print(f"  [{i:02d}] {step}")

Shape  : traingle
Sides  : 3
Side   : 2.0 m
Steps  : 6

  [00] distance_m=2.0 speed_ms=1.0 rate_hz=10.0 tool='move_forward'
  [01] angle_deg=120.0 angular_velocity_degs=90.0 rate_hz=10.0 tool='rotate'
  [02] distance_m=2.0 speed_ms=1.0 rate_hz=10.0 tool='move_forward'
  [03] angle_deg=120.0 angular_velocity_degs=90.0 rate_hz=10.0 tool='rotate'
  [04] distance_m=2.0 speed_ms=1.0 rate_hz=10.0 tool='move_forward'
  [05] angle_deg=120.0 angular_velocity_degs=90.0 rate_hz=10.0 tool='rotate'


Execute the JSON plan — iterate the steps and call the matching tool:


In [3]:
def execute_json_plan(plan: DrawingPlan, verbose: bool = True) -> list[str]:
    """Run each step in a DrawingPlan using the real tool suite."""
    observations = []
    for i, step in enumerate(plan.steps):
        tool_fn = TOOL_OBJECTS.get(step.tool)
        args = step.model_dump(exclude={"tool"})
        obs = safe_invoke(tool_fn, args) if tool_fn else f"Unknown tool: {step.tool}"
        observations.append(obs)
        if verbose:
            print(f"  [{i:02d}] {step.tool}({args}) → {obs}")
    return observations


print(f"Executing '{plan.shape}' plan ({len(plan.steps)} steps):\n")
# Let's reset
safe_invoke(TOOL_OBJECTS.get('reset'), {})
observations = execute_json_plan(plan)

Executing 'traingle' plan (6 steps):

  [00] move_forward({'distance_m': 2.0, 'speed_ms': 1.0, 'rate_hz': 10.0}) → Moved forward 2.00 m at 1.00 m/s (20 messages over 2.00 s). x=7.54, y=5.54, heading=0.0° (0.000 rad), linear_vel=0.00 m/s, angular_vel=0.00 rad/s
  [01] rotate({'angle_deg': 120.0, 'angular_velocity_degs': 90.0, 'rate_hz': 10.0}) → Rotated CCW 120.0° at 90.0 deg/s over 1.33 s (14 messages at 10 Hz). x=7.54, y=5.54, heading=121.0° (2.111 rad), linear_vel=0.00 m/s, angular_vel=0.00 rad/s
  [02] move_forward({'distance_m': 2.0, 'speed_ms': 1.0, 'rate_hz': 10.0}) → Moved forward 2.00 m at 1.00 m/s (20 messages over 2.00 s). x=6.52, y=7.26, heading=121.0° (2.111 rad), linear_vel=0.00 m/s, angular_vel=0.00 rad/s
  [03] rotate({'angle_deg': 120.0, 'angular_velocity_degs': 90.0, 'rate_hz': 10.0}) → Rotated CCW 120.0° at 90.0 deg/s over 1.33 s (14 messages at 10 Hz). x=6.52, y=7.26, heading=-118.1° (-2.061 rad), linear_vel=0.00 m/s, angular_vel=0.00 rad/s
  [04] move_forward({'dist

## Exercise 4c — Plan as code (code-as-actions)

The second approach asks the model to write a **Python script** instead of filling in a schema.
No Pydantic model required — the model uses its own knowledge of Python and geometry.

The key prompting technique: ask the model to wrap its code in XML tags.
This makes extraction unambiguous — no fence-stripping, no format guessing.

```
Write a Python script for the task below.
Wrap your code in <execute_python> </execute_python> tags.

<task>
Draw a square with side length 2 metres.
</task>
```

The model returns something like:

```
<execute_python>
for _ in range(4):
    move_forward(2)
    rotate(90.0)
</execute_python>
```

A square is one `for` loop. With the JSON schema approach it would be 12 individual steps.
Loops, variables, and `math` are all available — the model uses them naturally.


In [4]:
CODE_SYSTEM = """You are a Python code generator for a TurtleSim robot.
The turtle operates in an 11 x 11 metre arena, starting at x=5.5, y=5.5, heading=0 (East).

You may ONLY call these functions:
  move_forward(distance_m: float)  -- move forward by distance_m metres
  rotate(angle_deg: float)         -- rotate; positive = counter-clockwise
  get_pose()                       -- returns current (x, y, heading_deg)

Rules:
  - Use Python freely: variables, loops, and math are all allowed.
  - Do NOT import anything.
  - Do NOT call any function not listed above.
  - Wrap your code in <execute_python> </execute_python> tags.
"""

def generate_code(task: str, verbose: bool = True) -> str:
    """Ask the LLM to write a Python script, extracted from XML tags."""
    llm = get_llm()
    messages = [
        SystemMessage(CODE_SYSTEM),
        HumanMessage(f"<task>\n{task}\n</task>"),
    ]
    response = llm.invoke(messages)

    # Extract code from <execute_python> tags
    match = re.search(
        r"<execute_python>(.*?)</execute_python>",
        response.text,
        re.DOTALL,
    )
    if not match:
        raise ValueError(f"No <execute_python> tags found in response:\n{response.text}")

    code = match.group(1).strip()

    if verbose:
        print("Generated code:")
        print("─" * 50)
        print(code)
        print("─" * 50)

    return code


hex_code = generate_code("Draw a hexagon with side length 2 metres.")

Generated code:
──────────────────────────────────────────────────
for i in range(6):
    move_forward(2.0)
    rotate(60.0)
──────────────────────────────────────────────────


In [5]:
# bonus check
random_code = generate_code("Get the square root of any given number if valid.", verbose=False)
print(random_code)

# Given number to calculate the square root of
number = 4.0

# Check if the number is valid (non-negative)
if number >= 0:
    # Calculate the square root using the exponent operator
    sqrt_value = number ** 0.5
    # Use the result to move the robot forward
    move_forward(sqrt_value)


## Exercise 4d — Verify the generated code

Before running model-generated code, check two things with `ast` — no execution needed:
1. **Syntax** — is it valid Python?
2. **Safety** — does it only call approved functions?

This is the equivalent of validating a JSON plan's tool names and argument types,
but requires no schema — just a set of allowed function names.


In [ ]:
def verify_code(code: str) -> tuple[bool, str]:
    """Check syntax and function whitelist. Returns (ok, message)."""
    ALLOWED = set(TOOL_FUNCTIONS.keys()) | {"print", "range", "len", "abs", "round"}

    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return False, f"Syntax error: {e}"

    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            return False, "Unauthorised import statement"

    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
            if node.func.id not in ALLOWED:
                return False, f"Unauthorised call: '{node.func.id}'"

    n_calls = sum(1 for n in ast.walk(tree) if isinstance(n, ast.Call))
    return True, f"OK — {n_calls} call(s)"


# Test on the generated hexagon code
ok, msg = verify_code(hex_code)
print(f"Hexagon code : {'✓' if ok else '✗'}  {msg}")

# Test a hallucinated function name
ok, msg = verify_code("fly(10m)")
print(f"Hallucinated : {'✓' if ok else '✗'}  {msg}")

# Test an import injection
ok, msg = verify_code("import os\nos.system('echo hello')")
print(f"Import inject: {'✓' if ok else '✗'}  {msg}")

Hexagon code : ✓  OK — 3 call(s)
Hallucinated : ✗  Syntax error: invalid decimal literal (<unknown>, line 1)
Import inject: ✗  Unauthorised import statement


# Think
Why is it important to verify code before executing it?

## Exercise 4e — Execute safely + full pipeline

The full pipeline is: **generate → verify → execute**.
If verification fails, the LLM is asked to fix its code and we try again.


In [7]:
def execute_code(code: str, verbose: bool = True) -> list[str]:
    """Execute verified code"""

    observations = []
    def tracked(name, fn):
        def wrapper(*args, **kwargs):
            result = fn(*args, **kwargs)
            observations.append(f"{name}{args} → {result}")
            return result
        return wrapper

    safe_globals = {name: tracked(name, fn) for name, fn in TOOL_FUNCTIONS.items()}
    safe_globals["__builtins__"] = {
        "range": range, "len": len, "abs": abs,
        "round": round, "print": print, "math": math,
    }

    if verbose:
        print(f"── {'EXECUTING'} ──────────────────────────────")
    exec(code, safe_globals)
    if verbose:
        print(f"── {len(observations)} call(s) made ──────────────────────────────")
    return observations


def plan_and_execute(
    task: str,
    max_fix_attempts: int = 2,
    verbose: bool = True,
) -> dict:
    """Generate → verify → execute. Asks the LLM to fix on verification failure."""
    llm = get_llm()
    messages = [
        SystemMessage(CODE_SYSTEM),
        HumanMessage(f"<task>\n{task}\n</task>"),
    ]

    generated = None
    for attempt in range(1, max_fix_attempts + 2):
        if verbose:
            print(f"── Attempt {attempt} ────────────────────────────────────")

        response = llm.invoke(messages)
        match = re.search(r"<execute_python>(.*?)</execute_python>",
                          response.text, re.DOTALL)
        if not match:
            err = "No <execute_python> tags found."
            if verbose: print(f"   Extract: ✗  {err}")
        else:
            code = match.group(1).strip()
            ok, msg = verify_code(code)
            if verbose: print(f"   Verify : {'✓' if ok else '✗'}  {msg}")
            if ok:
                generated = code
                break
            err = msg

        messages.append(response)
        messages.append(HumanMessage(
            f"Your code failed: {err}\n"
            "Fix it and respond with <execute_python> tags only."
        ))

    if generated is None:
        return {"error": f"Could not produce valid code after {max_fix_attempts + 1} attempts."}

    if verbose:
        print()
        print("Final code:")
        print("─" * 50)
        print(generated)
        print("─" * 50)
        print()

    observations = execute_code(generated, verbose=verbose)
    return {"code": generated, "observations": observations}


# ── Run it ────────────────────────────────────────────────────────────────────
Task = "Draw a pentagon with side length 1 metres."
result = plan_and_execute(
    Task,
)

── Attempt 1 ────────────────────────────────────
   Verify : ✓  OK — 3 call(s)

Final code:
──────────────────────────────────────────────────
for i in range(5):
    move_forward(1.0)
    rotate(72.0)
──────────────────────────────────────────────────

── EXECUTING ──────────────────────────────
── 10 call(s) made ──────────────────────────────


### 📝 Activity — compare both approaches

Try the same shape with both planners and compare:

```python
# JSON schema approach
plan = planner.invoke([SystemMessage(SCHEMA_SYSTEM), HumanMessage("Draw a hexagon with side length 1.5 metres.")])
print(f"Steps in JSON plan: {len(plan.steps)}")

# Code approach
result = plan_and_execute("Draw a hexagon with side length 1.5 metres.")
print(f"Lines of code: {len(result['code'].splitlines())}")
```

**What to notice:**
- The JSON plan has 12 individual steps. The code is 2–3 lines with a loop.
- Both use exactly one LLM call.
- The JSON plan is easier to inspect step-by-step. The code is more expressive for complex shapes.


## Summary

**Approach 1 — JSON schema plan:**

| Step | How |
|---|---|
| Plan | `llm.with_structured_output(DrawingPlan).invoke(messages)` |
| Output | A validated Pydantic object — no parsing needed |
| Execute | Iterate `plan.steps`, call matching tool function |
| Best for | When you need a human-readable, inspectable, replayable plan |

**Approach 2 — Code-as-actions:**

| Step | How |
|---|---|
| Plan | `llm.invoke()` with XML tag instruction in system prompt |
| Extract | `re.search(r'<execute_python>(.*?)</execute_python>', ...)` |
| Verify | `ast.parse()` + function name whitelist |
| Execute | `exec(code, safe_globals)` with restricted builtins |
| Best for | Complex shapes, loops, math — anything flat JSON can't express cleanly |

**Both approaches:**
- Use **one LLM call** to plan
- Execute in **pure Python** — deterministic, fast, no LLM in the loop
- Can retry on failure by sending the error back to the model

Next: Coffee Break and sharing thoughts with sticky notes.